# Notebook 02 — Feature Engineering & Embeddings

**Requires:** `contracts_clean.parquet` produced by Notebook 01

### What this notebook does
1. Builds a TF-IDF matrix on cleaned contract descriptions
2. Scales structured (numeric) features
3. Encodes contracts with `paraphrase-multilingual-mpnet-base-v2` (768-dim)
4. Detects near-duplicate / copy-paste notices via cosine similarity
5. Extracts named entities (ORG, PER, LOC) with a NER model
6. Visualises the embedding space with t-SNE
7. Saves: `embeddings.npy`, `contract_ids.csv`, `X_tfidf.npz`, `X_struct.npy`, `ner_entities.csv`

### Works on
- ✅ Google Colab
- ✅ Local Jupyter

## ⚙️ Setup

In [ ]:
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in: {"Google Colab" if IN_COLAB else "Local Jupyter"}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # ⚠️  Same path as Notebook 01
    DRIVE_ROOT  = Path('/content/drive/MyDrive/NLP')
    RESULTS_DIR = DRIVE_ROOT / 'results'
    MODELS_DIR  = DRIVE_ROOT / 'models'
else:
    ROOT        = Path().resolve().parent
    RESULTS_DIR = ROOT / 'results'
    MODELS_DIR  = ROOT / 'models'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RESULTS_DIR / 'contracts_clean.parquet'
assert DATA_PATH.exists(), (
    f'contracts_clean.parquet not found at {DATA_PATH}.\n'
    'Run Notebook 01 first and make sure RESULTS_DIR points to the same folder.'
)
print(f'RESULTS_DIR = {RESULTS_DIR}')

In [ ]:
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', '-q',
                    'sentence-transformers', 'faiss-cpu',
                    'transformers', 'torch', 'pyarrow', 'seaborn'], check=True)
    print('Dependencies ready.')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set_theme(style='whitegrid')

df = pd.read_parquet(DATA_PATH)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

## 1. TF-IDF matrix (classical baseline)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

texts = df['text_clean'].fillna('').tolist()

tfidf = TfidfVectorizer(
    sublinear_tf=True,
    max_features=50_000,
    ngram_range=(1, 2),
    min_df=3,
    strip_accents='unicode',
)
X_tfidf = tfidf.fit_transform(texts)
print(f'TF-IDF matrix : {X_tfidf.shape}')
print(f'Sparsity      : {100*(1 - X_tfidf.nnz / (X_tfidf.shape[0]*X_tfidf.shape[1])):.2f}%')

joblib.dump(tfidf, MODELS_DIR / 'tfidf_vectorizer.joblib')
sp.save_npz(RESULTS_DIR / 'X_tfidf.npz', X_tfidf)
print('Saved: tfidf_vectorizer.joblib, X_tfidf.npz')

In [ ]:
# Top TF-IDF terms
feature_names = np.array(tfidf.get_feature_names_out())
mean_w = np.asarray(X_tfidf.mean(axis=0)).flatten()
top_idx = mean_w.argsort()[-30:][::-1]
print('Top 30 terms by mean TF-IDF weight:')
for term, w in zip(feature_names[top_idx], mean_w[top_idx]):
    print(f'  {term:<35} {w:.5f}')

## 2. Structured feature matrix

In [ ]:
from sklearn.preprocessing import StandardScaler

STRUCT_COLS = [
    'log_estimated_price', 'log_buyer_contract_count',
    'tender_days', 'is_open_procedure', 'is_negotiated', 'is_meat',
    'isFrameworkAgreement', 'isCoveredByGpa', 'isElectronicAuction',
    'isCentralProcurement', 'isJointProcurement',
    'score_transparency', 'score_administrative',
]
STRUCT_COLS = [c for c in STRUCT_COLS if c in df.columns]

X_struct = df[STRUCT_COLS].fillna(0).values.astype(np.float32)
scaler   = StandardScaler()
X_struct_scaled = scaler.fit_transform(X_struct)

joblib.dump(scaler, MODELS_DIR / 'struct_scaler.joblib')
np.save(RESULTS_DIR / 'X_struct.npy', X_struct_scaled)
print(f'Structured feature matrix: {X_struct_scaled.shape}')
print(f'Saved: struct_scaler.joblib, X_struct.npy')

In [ ]:
# Correlation heatmap vs. suspicious label
struct_df = pd.DataFrame(X_struct, columns=STRUCT_COLS)
struct_df['suspicious'] = df['suspicious'].astype(float)
corr = struct_df.corr()[['suspicious']].sort_values('suspicious', ascending=False)

fig, ax = plt.subplots(figsize=(5, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=ax, linewidths=0.5, cbar=False)
ax.set_title('Correlation with suspicious label')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'feature_correlation.png', dpi=120)
plt.show()

## 3. Multilingual sentence embeddings

Model: `paraphrase-multilingual-mpnet-base-v2`  
- Supports 50+ languages  
- 768-dimensional output  
- Downloaded once (~420 MB) and cached automatically

In [ ]:
from sentence_transformers import SentenceTransformer

ENCODER_NAME = 'paraphrase-multilingual-mpnet-base-v2'
encoder = SentenceTransformer(ENCODER_NAME)
print(f'Encoder loaded: {ENCODER_NAME}')
print(f'Embedding dim : {encoder.get_sentence_embedding_dimension()}')

In [ ]:
# Encode contracts
# MAX_ENCODE: increase to len(df) if running on GPU; keep at 50 000 for CPU
MAX_ENCODE = 50_000
texts_to_encode = df['text_clean'].fillna('').tolist()[:MAX_ENCODE]

print(f'Encoding {len(texts_to_encode):,} contracts…  (this takes a few minutes on CPU)')
embeddings = encoder.encode(
    texts_to_encode,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,   # unit-norm → cosine similarity = dot product
)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
np.save(RESULTS_DIR / 'embeddings.npy', embeddings)

contract_ids = df['id'].iloc[:MAX_ENCODE].reset_index(drop=True)
contract_ids.to_csv(RESULTS_DIR / 'contract_ids.csv', index=False)

print('Saved: embeddings.npy, contract_ids.csv')

## 4. Copy-paste / near-duplicate detection

In [ ]:
import faiss

emb_f32 = embeddings.astype(np.float32)
dim     = emb_f32.shape[1]

index = faiss.IndexFlatIP(dim)   # inner product on unit vectors = cosine similarity
index.add(emb_f32)
print(f'FAISS index: {index.ntotal:,} vectors, dim={dim}')

In [ ]:
SIMILARITY_THRESHOLD = 0.92   # contracts above this are likely copy-paste
K = 3

scores, indices = index.search(emb_f32, K)

near_dupes = []
for i, (nbr_scores, nbr_idxs) in enumerate(zip(scores, indices)):
    for score, j in zip(nbr_scores[1:], nbr_idxs[1:]):    # skip self
        if score >= SIMILARITY_THRESHOLD:
            near_dupes.append({
                'id_a':       df['id'].iloc[i],
                'id_b':       df['id'].iloc[j],
                'cosine_sim': float(score),
                'title_a':    df['title'].iloc[i],
                'title_b':    df['title'].iloc[j],
            })

dupes_df = pd.DataFrame(near_dupes).drop_duplicates(subset=['id_a','id_b'])
print(f'Near-duplicate pairs (cosine ≥ {SIMILARITY_THRESHOLD}): {len(dupes_df):,}')
dupes_df.to_csv(RESULTS_DIR / 'near_duplicates.csv', index=False)
dupes_df.head()

## 5. NER — extract company and buyer entities

In [ ]:
from transformers import pipeline as hf_pipeline

ner = hf_pipeline(
    'ner',
    model='dslim/bert-base-NER',
    aggregation_strategy='simple',
    device=-1,       # CPU; set to 0 for GPU
)
print('NER pipeline loaded.')

In [ ]:
NER_SAMPLE = 2000   # increase for full run
ner_sample = df['text_clean'].dropna().sample(min(NER_SAMPLE, len(df)), random_state=42)

ner_records = []
for idx, text in ner_sample.items():
    try:
        entities = ner(text[:512])
        for ent in entities:
            if ent['entity_group'] in ('ORG', 'PER', 'LOC'):
                ner_records.append({
                    'contract_id':  df['id'].loc[idx],
                    'entity_text':  ent['word'],
                    'entity_type':  ent['entity_group'],
                    'score':        round(ent['score'], 4),
                })
    except Exception:
        pass

ner_df = pd.DataFrame(ner_records)
ner_df.to_csv(RESULTS_DIR / 'ner_entities.csv', index=False)
print(f'Extracted {len(ner_df):,} entities from {NER_SAMPLE} contracts')
print('\nTop 20 ORG entities:')
print(ner_df[ner_df['entity_type']=='ORG']['entity_text'].value_counts().head(20).to_string())

## 6. t-SNE visualisation of embedding space

In [ ]:
from sklearn.manifold import TSNE

TSNE_N   = 3000
tsne_idx = np.random.default_rng(42).choice(len(embeddings), min(TSNE_N, len(embeddings)), replace=False)
emb_sub  = embeddings[tsne_idx]
labels_sub = df['suspicious'].astype(float).fillna(0.5).values[:len(embeddings)][tsne_idx]

print('Running t-SNE…')
tsne   = TSNE(n_components=2, perplexity=30, random_state=42, n_jobs=-1)
emb_2d = tsne.fit_transform(emb_sub)
print('Done.')

In [ ]:
color_map  = {0.0: '#2ecc71', 1.0: '#e74c3c', 0.5: '#bdc3c7'}
label_name = {0.0: 'Clean', 1.0: 'Suspicious', 0.5: 'Unknown'}

fig, ax = plt.subplots(figsize=(10, 7))
for val in [0.5, 0.0, 1.0]:
    mask = labels_sub == val
    if mask.sum():
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                   c=color_map[val], label=label_name[val],
                   alpha=0.5, s=8, rasterized=True)
ax.set_title(f't-SNE — multilingual contract embeddings (n={TSNE_N})')
ax.legend(markerscale=4)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'tsne_embeddings.png', dpi=120)
plt.show()

In [ ]:
# Add copy-paste flag back to parquet and re-save
copy_paste_ids = set(dupes_df['id_a']) | set(dupes_df['id_b'])
df['is_copy_paste'] = df['id'].isin(copy_paste_ids).astype(int)
df.to_parquet(RESULTS_DIR / 'contracts_clean.parquet', index=False)

print('\nNotebook 02 complete. Files saved to', RESULTS_DIR)
for f in ['embeddings.npy','contract_ids.csv','X_tfidf.npz',
          'X_struct.npy','near_duplicates.csv','ner_entities.csv']:
    p = RESULTS_DIR / f
    size = p.stat().st_size // 1024 if p.exists() else 0
    print(f'  {f:<35} {size:>8} KB')